In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torchvision.models import vit_b_16
from torchvision.models import ViT_B_16_Weights

from torch.utils.data import DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
TRAIN_DIR = "../data/COVID_19_dataset/train"
VAL_DIR   = "../data/COVID_19_dataset/val"
TEST_DIR  = "../data/COVID_19_dataset/test"

In [ ]:
def apply_clahe(img):

    img = np.array(img)

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_RGB2GRAY
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(gray)

    enhanced = cv2.cvtColor(
        enhanced,
        cv2.COLOR_GRAY2RGB
    )

    return Image.fromarray(
        enhanced
    )

In [ ]:
mean = [0.5159, 0.5159, 0.5159]
std  = [0.2280, 0.2280, 0.2280]
train_transform = transforms.Compose([

    transforms.Lambda(
        apply_clahe
    ),

    transforms.Resize(
        (224,224)
    ),

    transforms.RandomAffine(
        degrees=5,
        translate=(0.03,0.03)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])
test_transform = transforms.Compose([

    transforms.Lambda(
        apply_clahe
    ),

    transforms.Resize(
        (224,224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

In [ ]:
CLASS_NAMES = train_dataset.classes

print(CLASS_NAMES)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
weights = ViT_B_16_Weights.IMAGENET1K_V1

model = vit_b_16(
    weights=weights
)

In [ ]:
for param in model.parameters():

    param.requires_grad = False
for param in model.heads.parameters():

    param.requires_grad = True


In [ ]:
model.heads.head = nn.Linear(
    768,
    3
)

In [ ]:
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(

    model.heads.parameters(),

    lr=1e-3
)

In [ ]:
def train_one_epoch():

    model.train()

    running_loss = 0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(
            outputs,
            1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(train_loader)

    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
def validate():

    model.eval()

    running_loss = 0

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += loss.item()

            _, preds = torch.max(
                outputs,
                1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    val_loss = running_loss / len(val_loader)

    val_acc = correct / total

    return val_loss, val_acc

In [ ]:
EPOCHS = 10
history = {

    "train_loss": [],
    "train_acc": [],

    "val_loss": [],
    "val_acc": []
}
for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Accuracy: {train_acc:.4f}"
    )

    print(
        f"Validation Loss: {val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: {val_acc:.4f}"
    )

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_range,
    history['train_loss'],
    marker='o',
    label='Train Loss'
)

plt.plot(
    epochs_range,
    history['val_loss'],
    marker='o',
    label='Validation Loss'
)

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("ViT-B/16 Loss Curve")
plt.legend()
plt.grid(True)

plt.savefig(
    "../outputs/vit/vit_loss_curve.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()



In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    epochs_range,
    history['train_acc'],
    marker='o',
    label='Train Accuracy'
)

plt.plot(
    epochs_range,
    history['val_acc'],
    marker='o',
    label='Validation Accuracy'
)

plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("ViT-B/16 Accuracy Curve")
plt.legend()
plt.grid(True)

plt.savefig(
    "../outputs/vit/vit_accuracy_curve.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
torch.save(

    model.state_dict(),

    "vit_b16.pth"
)

print("ViT-B/16 model saved.")

In [ ]:
import json

metrics = {
    "final_train_accuracy": float(history['train_acc'][-1]),
    "final_validation_accuracy": float(history['val_acc'][-1]),
    "final_train_loss": float(history['train_loss'][-1]),
    "final_validation_loss": float(history['val_loss'][-1])
}

with open(
    "../outputs/vit/vit_metrics.json",
    "w"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )

print("Metrics saved!")

In [ ]:
config = {
    "model": "ViT-B/16",
    "image_size": 224,
    "batch_size": 16,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "frozen_backbone": True
}

with open(
    "../outputs/vit/vit_config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

print("Config saved!")

In [ ]:
import torch.nn.functional as F

model.eval()

predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = F.softmax(outputs, dim=1)

        confidences, preds = torch.max(probs, 1)

        for i in range(len(images)):

            predictions.append({
                "true_label": CLASS_NAMES[labels[i].item()],
                "predicted_label": CLASS_NAMES[preds[i].item()],
                "confidence": float(confidences[i].item()),
                "correct": bool(
                    labels[i] == preds[i]
                )
            })

pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    "../outputs/vit/vit_predictions.csv",
    index=False
)

print("Predictions CSV saved!")

In [ ]:
report = classification_report(
    pred_df["true_label"],
    pred_df["predicted_label"]
)

with open(
    "../outputs/vit/vit_classification_report.txt",
    "w"
) as f:

    f.write(report)

print(report)

In [ ]:
import seaborn as sns 
cm = confusion_matrix(
    pred_df["true_label"],
    pred_df["predicted_label"]
)

np.save(
    "../outputs/vit/vit_confusion_matrix.npy",
    cm
)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("ViT-B/16 Confusion Matrix")

plt.savefig(
    "../outputs/vit/vit_confusion_matrix.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()